# Skin Cancer MNIST: HAM10000 with recent architectures (PyTorch)

Same sections as the original notebook, rebuilt for the improved manuscript. Changes: ten models (InceptionV3, MobileNetV2, EfficientNet-B0, EfficientNetV2-S, ConvNeXt-T, Swin-T, ViT-S/16, DINOv2 ViT-S/14, baseline CNN, CNN-LSTM); a **lesion-grouped** split of all images (all images of a lesion stay in one partition); class-weighted loss instead of pre-generated augmented copies; balanced accuracy, macro-F1, MCC, AUROC and bootstrap intervals instead of accuracy and micro-F1 alone; Grad-CAM; resumable training.

**Setup:** Add Input -> *Skin Cancer MNIST: HAM10000*; Accelerator = **GPU**; Internet = **On**. For long runs use **Save Version -> Save & Run All (Commit)** so the run continues in the background and its output is kept.

In [ ]:
# Write the project source files (src/*.py) into the working directory
import os
FILES = {
"src/__init__.py": "",
"src/common.py": "\"\"\"Shared constants for the HAM10000 skin-lesion classification pipeline.\"\"\"\n\nCLASS_CODES = [\"akiec\", \"bcc\", \"bkl\", \"df\", \"mel\", \"nv\", \"vasc\"]\nCLASS_NAMES = {\n    \"akiec\": \"Actinic keratoses / intraepithelial carcinoma\",\n    \"bcc\": \"Basal cell carcinoma\",\n    \"bkl\": \"Benign keratosis-like lesions\",\n    \"df\": \"Dermatofibroma\",\n    \"mel\": \"Melanoma\",\n    \"nv\": \"Melanocytic nevi\",\n    \"vasc\": \"Vascular lesions\",\n}\nNUM_CLASSES = len(CLASS_CODES)\nCODE_TO_IDX = {c: i for i, c in enumerate(CLASS_CODES)}\n\n# Models evaluated in the study, in the order used in tables and figures.\nMODEL_ORDER = [\"simple_cnn\", \"cnn_lstm\", \"mobilenetv2\", \"inceptionv3\", \"efficientnet_b0\",\n               \"efficientnet_v2_s\", \"convnext_tiny\", \"swin_t\", \"vit_small\", \"dinov2_small\"]\nMODEL_LABELS = {\n    \"simple_cnn\": \"Baseline CNN\",\n    \"cnn_lstm\": \"CNN-LSTM\",\n    \"mobilenetv2\": \"MobileNetV2\",\n    \"inceptionv3\": \"InceptionV3\",\n    \"efficientnet_b0\": \"EfficientNet-B0\",\n    \"vit_small\": \"ViT-S/16\",\n    \"efficientnet_v2_s\": \"EfficientNetV2-S\",\n    \"convnext_tiny\": \"ConvNeXt-T\",\n    \"swin_t\": \"Swin-T\",\n    \"dinov2_small\": \"DINOv2 ViT-S/14\",\n}\n",
"src/data.py": "\"\"\"Data loading, lesion-level splitting and image transforms for HAM10000.\"\"\"\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nfrom sklearn.model_selection import StratifiedGroupKFold\n\nfrom .common import CODE_TO_IDX\n\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\n\n\ndef index_images(data_dir):\n    \"\"\"Map image_id -> file path for every .jpg below data_dir.\"\"\"\n    paths = {p.stem: str(p) for p in Path(data_dir).rglob(\"*.jpg\")}\n    if not paths:\n        raise FileNotFoundError(f\"No .jpg files found below {data_dir}\")\n    return paths\n\n\ndef load_metadata(data_dir):\n    meta = pd.read_csv(Path(data_dir) / \"HAM10000_metadata.csv\")\n    paths = index_images(data_dir)\n    meta = meta[meta[\"image_id\"].isin(paths)].copy()\n    meta[\"path\"] = meta[\"image_id\"].map(paths)\n    meta[\"label\"] = meta[\"dx\"].map(CODE_TO_IDX)\n    return meta.reset_index(drop=True)\n\n\ndef make_splits(meta, seed=42, n_folds=10, test_folds=2, val_folds=1):\n    \"\"\"Stratified, lesion-grouped split into train/val/test (about 70/10/20).\n\n    HAM10000 contains several images of the same lesion. Grouping by\n    lesion_id keeps all images of a lesion in a single partition and prevents\n    near-duplicate leakage between training and evaluation data.\n    \"\"\"\n    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)\n    fold = np.full(len(meta), -1)\n    for k, (_, idx) in enumerate(sgkf.split(meta, meta[\"label\"], groups=meta[\"lesion_id\"])):\n        fold[idx] = k\n    split = np.where(fold < test_folds, \"test\",\n                     np.where(fold < test_folds + val_folds, \"val\", \"train\"))\n    out = meta.copy()\n    out[\"split\"] = split\n    return out\n\n\ndef build_transforms(size, train):\n    from torchvision import transforms as T\n    norm = T.Normalize(IMAGENET_MEAN, IMAGENET_STD)\n    if train:\n        return T.Compose([\n            T.RandomResizedCrop(size, scale=(0.7, 1.0), ratio=(0.9, 1.1)),\n            T.RandomHorizontalFlip(),\n            T.RandomVerticalFlip(),\n            T.RandomRotation(30),\n            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),\n            T.ToTensor(),\n            norm,\n        ])\n    return T.Compose([T.Resize((size, size)), T.ToTensor(), norm])\n\n\nclass LesionDataset:\n    \"\"\"Minimal map-style dataset (compatible with torch.utils.data.DataLoader).\"\"\"\n\n    def __init__(self, frame, transform):\n        self.paths = frame[\"path\"].tolist()\n        self.labels = frame[\"label\"].astype(int).tolist()\n        self.transform = transform\n\n    def __len__(self):\n        return len(self.paths)\n\n    def __getitem__(self, i):\n        img = Image.open(self.paths[i]).convert(\"RGB\")\n        return self.transform(img), self.labels[i]\n",
"src/diagrams.py": "\"\"\"Schematic figures (study workflow and InceptionV3 layout); no experimental data involved.\"\"\"\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import FancyBboxPatch\n\nfrom .report import TEXT_WIDTH_IN, save_fig\n\n\ndef _box(ax, x, y, w, h, text, fc=\"#eaf2f8\", ec=\"#1b4f72\", fs=7):\n    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=\"round,pad=0.02,rounding_size=0.04\",\n                                fc=fc, ec=ec, lw=0.9))\n    ax.text(x + w / 2, y + h / 2, text, ha=\"center\", va=\"center\", fontsize=fs)\n\n\ndef _arrow(ax, x0, y0, x1, y1):\n    ax.annotate(\"\", xy=(x1, y1), xytext=(x0, y0),\n                arrowprops=dict(arrowstyle=\"-|>\", color=\"#333333\", lw=0.9))\n\n\ndef workflow(figdir):\n    fig, ax = plt.subplots(figsize=(TEXT_WIDTH_IN, 2.9))\n    ax.set_xlim(0, 12); ax.set_ylim(0, 5.8); ax.axis(\"off\"); ax.grid(False)\n    top = [(\"HAM10000\\n10,015 images,\\n7 classes\", 0.1),\n           (\"Lesion-level\\nstratified split\\n(train / val / test)\", 3.1),\n           (\"Resizing,\\nnormalisation,\\naugmentation\", 6.1),\n           (\"Backbone\\n(transfer\\nlearning)\", 9.1)]\n    for t, x in top:\n        _box(ax, x, 3.6, 2.4, 1.7, t, fs=6)\n    for i in range(3):\n        _arrow(ax, top[i][1] + 2.4, 4.45, top[i + 1][1], 4.45)\n    bottom = [(\"Class-weighted\\nloss, AdamW,\\nearly stopping\", 9.1),\n              (\"Model selection:\\nvalidation\\nmacro-F1\", 6.1),\n              (\"Test evaluation:\\nmetrics, CI,\\nROC curves\", 3.1),\n              (\"Tables and\\nfigures\", 0.1)]\n    for t, x in bottom:\n        _box(ax, x, 0.5, 2.4, 1.7, t, fc=\"#fef5e7\", ec=\"#b9770e\", fs=6)\n    _arrow(ax, 10.3, 3.6, 10.3, 2.2)\n    for i in range(3):\n        _arrow(ax, bottom[i][1], 1.35, bottom[i + 1][1] + 2.4, 1.35)\n    save_fig(fig, Path(figdir) / \"fig_workflow\")\n\n\ndef inception(figdir):\n    fig, ax = plt.subplots(figsize=(TEXT_WIDTH_IN, 2.4))\n    ax.set_xlim(0, 12); ax.set_ylim(0, 4.6); ax.axis(\"off\"); ax.grid(False)\n    blocks = [(\"Input\\n299 x 299 x 3\", \"#f2f3f4\"), (\"Stem\\n(conv + pool)\", \"#eaf2f8\"),\n              (\"3 x\\nInception-A\", \"#d6eaf8\"), (\"Reduction\\nblock A\", \"#fdebd0\"),\n              (\"4 x\\nInception-B\", \"#d6eaf8\"), (\"Reduction\\nblock B\", \"#fdebd0\"),\n              (\"2 x\\nInception-C\", \"#d6eaf8\"), (\"Global\\naverage pool\", \"#e8f6f3\"),\n              (\"Dropout\", \"#fadbd8\"), (\"Linear\\n(7 classes)\", \"#fadbd8\")]\n    w, h, step = 1.9, 1.3, 2.45\n    pos = []\n    for i, (t, c) in enumerate(blocks):\n        row, col = divmod(i, 5)\n        x, y = 0.1 + col * step, 3.0 - row * 2.3\n        pos.append((x, y))\n        _box(ax, x, y, w, h, t, fc=c, fs=6.5)\n        if col:\n            _arrow(ax, x - (step - w), y + h / 2, x, y + h / 2)\n    _arrow(ax, pos[4][0] + w / 2, pos[4][1], pos[5][0] + w / 2, pos[5][1] + h)\n    save_fig(fig, Path(figdir) / \"fig_inception\")\n\n\nif __name__ == \"__main__\":\n    out = Path(\"paper/figures\")\n    workflow(out)\n    inception(out)\n    print(\"Diagrams written to\", out)\n",
"src/evaluate.py": "\"\"\"Metrics, bootstrap confidence intervals and trivial baselines (NumPy / scikit-learn only).\"\"\"\nimport numpy as np\nfrom sklearn.metrics import (balanced_accuracy_score, cohen_kappa_score, confusion_matrix,\n                             f1_score, matthews_corrcoef, precision_recall_fscore_support,\n                             roc_auc_score)\n\nfrom .common import NUM_CLASSES\n\n\ndef compute_metrics(y_true, probs, with_auc=True):\n    \"\"\"Overall and per-class metrics from true labels and class probabilities.\"\"\"\n    y_true = np.asarray(y_true)\n    probs = np.asarray(probs)\n    labels = list(range(NUM_CLASSES))\n    y_pred = probs.argmax(1)\n    cm = confusion_matrix(y_true, y_pred, labels=labels)\n    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)\n    tn = cm.sum() - cm.sum(0) - cm.sum(1) + np.diag(cm)\n    fp = cm.sum(0) - np.diag(cm)\n    spec = np.divide(tn, tn + fp, out=np.zeros(len(labels)), where=(tn + fp) > 0)\n    if not with_auc:\n        auc_macro, auc_each = float(\"nan\"), [float(\"nan\")] * len(labels)\n    else:\n        try:\n            auc_macro = float(roc_auc_score(y_true, probs, multi_class=\"ovr\", average=\"macro\", labels=labels))\n            auc_each = [float(roc_auc_score((y_true == k).astype(int), probs[:, k])) for k in labels]\n        except ValueError:\n            auc_macro, auc_each = float(\"nan\"), [float(\"nan\")] * len(labels)\n    return {\n        \"accuracy\": float((y_pred == y_true).mean()),\n        \"balanced_accuracy\": float(balanced_accuracy_score(y_true, y_pred)),\n        \"macro_f1\": float(f1_score(y_true, y_pred, labels=labels, average=\"macro\", zero_division=0)),\n        \"weighted_f1\": float(f1_score(y_true, y_pred, labels=labels, average=\"weighted\", zero_division=0)),\n        \"macro_precision\": float(p.mean()),\n        \"macro_recall\": float(r.mean()),\n        \"mcc\": float(matthews_corrcoef(y_true, y_pred)),\n        \"kappa\": float(cohen_kappa_score(y_true, y_pred)),\n        \"auroc_macro\": auc_macro,\n        \"per_class\": {\n            \"precision\": p.tolist(), \"recall\": r.tolist(), \"f1\": f.tolist(),\n            \"support\": s.astype(int).tolist(), \"specificity\": spec.tolist(), \"auroc\": auc_each,\n        },\n        \"confusion_matrix\": cm.tolist(),\n        \"n\": int(len(y_true)),\n    }\n\n\ndef bootstrap_ci(y_true, probs, keys=(\"accuracy\", \"balanced_accuracy\", \"macro_f1\", \"mcc\"),\n                 n_boot=1000, seed=0, alpha=0.05):\n    \"\"\"Percentile bootstrap confidence intervals over test images.\"\"\"\n    rng = np.random.default_rng(seed)\n    y_true = np.asarray(y_true)\n    probs = np.asarray(probs)\n    n = len(y_true)\n    draws = {k: [] for k in keys}\n    for _ in range(n_boot):\n        idx = rng.integers(0, n, n)\n        m = compute_metrics(y_true[idx], probs[idx], with_auc=False)\n        for k in keys:\n            draws[k].append(m[k])\n    lo, hi = 100 * alpha / 2, 100 * (1 - alpha / 2)\n    return {k: [float(np.percentile(v, lo)), float(np.percentile(v, hi))] for k, v in draws.items()}\n\n\ndef majority_baseline(y_train, y_test):\n    \"\"\"Predict the most frequent training class for every test image.\"\"\"\n    y_train = np.asarray(y_train)\n    y_test = np.asarray(y_test)\n    top = int(np.bincount(y_train, minlength=NUM_CLASSES).argmax())\n    probs = np.zeros((len(y_test), NUM_CLASSES))\n    probs[:, top] = 1.0\n    return compute_metrics(y_test, probs)\n",
"src/gradcam.py": "\"\"\"Grad-CAM saliency maps for the convolutional models.\n\nExample:\n    python -m src.gradcam --data-dir data/HAM10000 --model inceptionv3 \\\n        --checkpoint outputs/inceptionv3/seed0/best.pt --out paper/figures/fig_gradcam\n\"\"\"\nimport argparse\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\nfrom PIL import Image\n\nfrom .common import CLASS_CODES\nfrom .data import IMAGENET_MEAN, IMAGENET_STD, build_transforms\nfrom .models import INPUT_SIZE, build_model\n\nTARGET_LAYER = {\n    \"inceptionv3\": lambda m: m.Mixed_7c,\n    \"mobilenetv2\": lambda m: m.features[-1],\n    \"efficientnet_b0\": lambda m: m.features[-1],\n    \"efficientnet_v2_s\": lambda m: m.features[-1],\n    \"convnext_tiny\": lambda m: m.features[-1],\n    \"simple_cnn\": lambda m: m.features[-1],\n}\n\n\ndef grad_cam(model, layer, x, class_idx=None):\n    acts, grads = [], []\n    h1 = layer.register_forward_hook(lambda m, i, o: acts.append(o))\n    h2 = layer.register_full_backward_hook(lambda m, gi, go: grads.append(go[0]))\n    model.eval()\n    logits = model(x)\n    cls = int(logits.argmax(1)) if class_idx is None else class_idx\n    model.zero_grad()\n    logits[0, cls].backward()\n    h1.remove()\n    h2.remove()\n    weights = grads[0].mean(dim=(2, 3), keepdim=True)\n    cam = F.relu((weights * acts[0]).sum(1, keepdim=True))\n    cam = F.interpolate(cam, size=x.shape[-2:], mode=\"bilinear\", align_corners=False)[0, 0]\n    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)\n    return cam.detach().cpu().numpy(), cls\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--data-dir\", required=True)\n    ap.add_argument(\"--splits\", default=\"outputs/splits.csv\")\n    ap.add_argument(\"--model\", required=True, choices=list(TARGET_LAYER))\n    ap.add_argument(\"--checkpoint\", required=True)\n    ap.add_argument(\"--per-class\", type=int, default=1, help=\"test images shown per class\")\n    ap.add_argument(\"--out\", default=\"paper/figures/fig_gradcam\")\n    args = ap.parse_args()\n\n    frame = pd.read_csv(args.splits)\n    frame = frame[frame.split == \"test\"]\n    model, _ = build_model(args.model, pretrained=False)\n    model.load_state_dict(torch.load(args.checkpoint, map_location=\"cpu\"))\n    layer = TARGET_LAYER[args.model](model)\n    size = INPUT_SIZE[args.model]\n    tf = build_transforms(size, train=False)\n    mean = np.array(IMAGENET_MEAN)[:, None, None]\n    std = np.array(IMAGENET_STD)[:, None, None]\n\n    rows = []\n    for k, code in enumerate(CLASS_CODES):\n        rows += list(frame[frame.label == k].sample(args.per_class, random_state=0).itertuples())\n    fig, axes = plt.subplots(2, len(rows), figsize=(1.7 * len(rows), 3.6))\n    for j, r in enumerate(rows):\n        x = tf(Image.open(r.path).convert(\"RGB\"))[None].requires_grad_(True)\n        cam, pred = grad_cam(model, layer, x)\n        img = np.clip((x[0].detach().numpy() * std + mean).transpose(1, 2, 0), 0, 1)\n        axes[0, j].imshow(img)\n        axes[0, j].set_title(f\"true: {CLASS_CODES[r.label]}\\npred: {CLASS_CODES[pred]}\", fontsize=7)\n        axes[1, j].imshow(img)\n        axes[1, j].imshow(cam, cmap=\"jet\", alpha=0.45)\n        for a in axes[:, j]:\n            a.axis(\"off\")\n    plt.tight_layout()\n    out = Path(args.out)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    for ext in (\"pdf\", \"eps\", \"png\"):\n        fig.savefig(f\"{out}.{ext}\", dpi=600, bbox_inches=\"tight\")\n    print(\"Saved\", out)\n\n\nif __name__ == \"__main__\":\n    main()\n",
"src/models.py": "\"\"\"Model zoo: pretrained backbones and two from-scratch baselines.\"\"\"\nimport torch.nn as nn\nfrom torchvision import models\n\nfrom .common import NUM_CLASSES\n\nINPUT_SIZE = {\"inceptionv3\": 299, \"mobilenetv2\": 224, \"efficientnet_b0\": 224, \"efficientnet_v2_s\": 224,\n              \"convnext_tiny\": 224, \"swin_t\": 224, \"vit_small\": 224, \"dinov2_small\": 224,\n              \"simple_cnn\": 224, \"cnn_lstm\": 224}\nPRETRAINED = {\"inceptionv3\", \"mobilenetv2\", \"efficientnet_b0\", \"efficientnet_v2_s\", \"convnext_tiny\",\n              \"swin_t\", \"vit_small\", \"dinov2_small\"}\n\n\ndef _block(cin, cout):\n    return nn.Sequential(\n        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),\n        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),\n        nn.MaxPool2d(2),\n    )\n\n\nclass SimpleCNN(nn.Module):\n    \"\"\"Four-stage convolutional network trained from scratch.\"\"\"\n\n    def __init__(self, num_classes=NUM_CLASSES):\n        super().__init__()\n        self.features = nn.Sequential(_block(3, 32), _block(32, 64), _block(64, 128), _block(128, 256))\n        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.4),\n                                  nn.Linear(256, num_classes))\n\n    def forward(self, x):\n        return self.head(self.features(x))\n\n\nclass CNNLSTM(nn.Module):\n    \"\"\"Convolutional stem whose feature-map rows are read as a sequence by an LSTM.\"\"\"\n\n    def __init__(self, num_classes=NUM_CLASSES, hidden=256):\n        super().__init__()\n        self.features = nn.Sequential(_block(3, 32), _block(32, 64), _block(64, 128))\n        self.pool = nn.AdaptiveAvgPool2d((14, 14))\n        self.lstm = nn.LSTM(input_size=128 * 14, hidden_size=hidden, batch_first=True)\n        self.head = nn.Sequential(nn.Dropout(0.4), nn.Linear(hidden, num_classes))\n\n    def forward(self, x):\n        f = self.pool(self.features(x))                    # B, C, H, W\n        b, c, h, w = f.shape\n        seq = f.permute(0, 2, 1, 3).reshape(b, h, c * w)   # B, H, C*W\n        out, _ = self.lstm(seq)\n        return self.head(out[:, -1])\n\n\ndef build_model(name, pretrained=True, num_classes=NUM_CLASSES):\n    \"\"\"Return (model, head_module); head_module is trained first when fine-tuning.\"\"\"\n    if name == \"inceptionv3\":\n        w = models.Inception_V3_Weights.IMAGENET1K_V1 if pretrained else None\n        m = models.inception_v3(weights=w, aux_logits=True, transform_input=True, init_weights=not pretrained)\n        m.aux_logits, m.AuxLogits = False, None\n        m.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(m.fc.in_features, num_classes))\n        return m, m.fc\n    if name == \"mobilenetv2\":\n        w = models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None\n        m = models.mobilenet_v2(weights=w)\n        m.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(m.last_channel, num_classes))\n        return m, m.classifier\n    if name == \"efficientnet_b0\":\n        w = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None\n        m = models.efficientnet_b0(weights=w)\n        m.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(m.classifier[1].in_features, num_classes))\n        return m, m.classifier\n    if name == \"efficientnet_v2_s\":\n        w = models.EfficientNet_V2_S_Weights.IMAGENET1K_V1 if pretrained else None\n        m = models.efficientnet_v2_s(weights=w)\n        m.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(m.classifier[1].in_features, num_classes))\n        return m, m.classifier\n    if name == \"convnext_tiny\":\n        w = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None\n        m = models.convnext_tiny(weights=w)\n        m.classifier[2] = nn.Sequential(nn.Dropout(0.4), nn.Linear(m.classifier[2].in_features, num_classes))\n        return m, m.classifier\n    if name == \"swin_t\":\n        w = models.Swin_T_Weights.IMAGENET1K_V1 if pretrained else None\n        m = models.swin_t(weights=w)\n        m.head = nn.Sequential(nn.Dropout(0.4), nn.Linear(m.head.in_features, num_classes))\n        return m, m.head\n    if name == \"vit_small\":\n        import timm\n        m = timm.create_model(\"vit_small_patch16_224\", pretrained=pretrained, num_classes=num_classes,\n                              drop_rate=0.4)\n        return m, m.head\n    if name == \"dinov2_small\":\n        import timm\n        m = timm.create_model(\"vit_small_patch14_dinov2.lvd142m\", pretrained=pretrained,\n                              num_classes=num_classes, img_size=224, drop_rate=0.4)\n        return m, m.head\n    if name == \"simple_cnn\":\n        m = SimpleCNN(num_classes)\n        return m, m.head\n    if name == \"cnn_lstm\":\n        m = CNNLSTM(num_classes)\n        return m, m.head\n    raise ValueError(f\"Unknown model: {name}\")\n\n\ndef count_parameters(model):\n    return sum(p.numel() for p in model.parameters())\n",
"src/report.py": "\"\"\"Build all manuscript figures (EPS/PDF/PNG, 600 dpi) and LaTeX tables from saved results.\n\nExample:\n    python -m src.report --out-dir outputs --paper-dir paper --data-dir data/HAM10000\n\"\"\"\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_curve\n\nfrom .common import CLASS_CODES, CLASS_NAMES, MODEL_LABELS, MODEL_ORDER, NUM_CLASSES\n\nTEXT_WIDTH_IN = 12.2 / 2.54          # JAMI typing-area width\nplt.rcParams.update({\n    \"font.family\": \"serif\", \"font.size\": 8, \"axes.titlesize\": 8, \"axes.labelsize\": 8,\n    \"legend.fontsize\": 7, \"xtick.labelsize\": 7, \"ytick.labelsize\": 7,\n    \"pdf.fonttype\": 42, \"ps.fonttype\": 42, \"axes.grid\": True, \"grid.alpha\": 0.3, \"figure.dpi\": 150,\n})\nPALETTE = [\"#1b4f72\", \"#b9770e\", \"#1e8449\", \"#922b21\", \"#6c3483\", \"#117a8b\", \"#5d6d7e\"]\n\n\ndef save_fig(fig, base):\n    base = Path(base)\n    base.parent.mkdir(parents=True, exist_ok=True)\n    for ext in (\"eps\", \"pdf\", \"png\"):\n        fig.savefig(f\"{base}.{ext}\", dpi=600, bbox_inches=\"tight\")\n    plt.close(fig)\n\n\ndef load_runs(out_dir):\n    runs = {}\n    for m in MODEL_ORDER:\n        seeds = sorted((Path(out_dir) / m).glob(\"seed*\"))\n        res = []\n        for s in seeds:\n            if (s / \"metrics.json\").exists():\n                res.append({\"metrics\": json.loads((s / \"metrics.json\").read_text()),\n                            \"history\": json.loads((s / \"history.json\").read_text()),\n                            \"preds\": np.load(s / \"test_predictions.npz\")})\n        if res:\n            runs[m] = res\n    return runs\n\n\ndef ms(values, digits=3):\n    v = np.asarray(values, dtype=float)\n    if len(v) == 1:\n        return f\"${v[0]:.{digits}f}$\"\n    return f\"${v.mean():.{digits}f} \\\\pm {v.std(ddof=1) if len(v) > 1 else 0.0:.{digits}f}$\"\n\n\ndef write(path, text):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(text)\n\n\n# --------------------------------------------------------------------------- tables\ndef table_dataset(frame, gen):\n    rows = []\n    for k, code in enumerate(CLASS_CODES):\n        sub = frame[frame.label == k]\n        rows.append(f\"{code} & {CLASS_NAMES[code]} & {len(sub)} & \"\n                    f\"{(sub.split == 'train').sum()} & {(sub.split == 'val').sum()} & {(sub.split == 'test').sum()} \\\\\\\\\")\n    rows.append(\"\\\\hline\")\n    rows.append(f\"Total & & {len(frame)} & {(frame.split == 'train').sum()} & \"\n                f\"{(frame.split == 'val').sum()} & {(frame.split == 'test').sum()} \\\\\\\\\")\n    write(gen / \"table_dataset.tex\",\n          \"\\\\begin{tabular}{llrrrr}\\n\\\\hline\\nCode & Diagnostic category & All & Train & Val. & Test \\\\\\\\\\n\\\\hline\\n\"\n          + \"\\n\".join(rows) + \"\\n\\\\hline\\n\\\\end{tabular}\\n\")\n\n\ndef table_overall(runs, gen):\n    keys = [(\"accuracy\", \"Accuracy\"), (\"balanced_accuracy\", \"Bal. acc.\"), (\"macro_f1\", \"Macro-F1\"),\n            (\"mcc\", \"MCC\"), (\"auroc_macro\", \"AUROC\")]\n    rows = []\n    for m, res in runs.items():\n        cells = [ms([r[\"metrics\"][k] for r in res]) for k, _ in keys]\n        rows.append(f\"{MODEL_LABELS[m]} & \" + \" & \".join(cells) + \" \\\\\\\\\")\n    base = next(iter(runs.values()))[0][\"metrics\"][\"majority_baseline\"]\n    rows.append(\"\\\\hline\")\n    rows.append(\"Majority-class baseline & \" + \" & \".join(\n        f\"${base[k]:.3f}$\" if k != \"auroc_macro\" else \"$0.500$\" for k, _ in keys) + \" \\\\\\\\\")\n    write(gen / \"table_overall.tex\",\n          \"\\\\begin{tabular}{lccccc}\\n\\\\hline\\nModel & \" + \" & \".join(n for _, n in keys) + \" \\\\\\\\\\n\\\\hline\\n\"\n          + \"\\n\".join(rows) + \"\\n\\\\hline\\n\\\\end{tabular}\\n\")\n\n\ndef table_perclass(runs, primary, gen):\n    res = runs[primary]\n    pc = {k: np.mean([r[\"metrics\"][\"per_class\"][k] for r in res], axis=0)\n          for k in (\"precision\", \"recall\", \"specificity\", \"f1\", \"auroc\")}\n    support = res[0][\"metrics\"][\"per_class\"][\"support\"]\n    rows = [f\"{CLASS_CODES[i]} & {pc['precision'][i]:.3f} & {pc['recall'][i]:.3f} & {pc['specificity'][i]:.3f} & \"\n            f\"{pc['f1'][i]:.3f} & {pc['auroc'][i]:.3f} & {support[i]} \\\\\\\\\" for i in range(NUM_CLASSES)]\n    write(gen / \"table_perclass.tex\",\n          \"\\\\begin{tabular}{lcccccr}\\n\\\\hline\\nClass & Precision & Sensitivity & Specificity & F1 & AUROC & Support \\\\\\\\\\n\\\\hline\\n\"\n          + \"\\n\".join(rows) + \"\\n\\\\hline\\n\\\\end{tabular}\\n\")\n\n\ndef table_ci(runs, gen):\n    rows = []\n    for m, res in runs.items():\n        ci = res[0][\"metrics\"][\"ci95\"]\n        mt = res[0][\"metrics\"]\n        cells = [f\"${mt[k]:.3f}$ [{ci[k][0]:.3f}, {ci[k][1]:.3f}]\" for k in (\"accuracy\", \"balanced_accuracy\", \"macro_f1\", \"mcc\")]\n        rows.append(f\"{MODEL_LABELS[m]} & \" + \" & \".join(cells) + \" \\\\\\\\\")\n    write(gen / \"table_ci.tex\",\n          \"\\\\begin{tabular}{lcccc}\\n\\\\hline\\nModel & Accuracy & Bal. acc. & Macro-F1 & MCC \\\\\\\\\\n\\\\hline\\n\"\n          + \"\\n\".join(rows) + \"\\n\\\\hline\\n\\\\end{tabular}\\n\")\n\n\ndef table_cost(runs, gen):\n    rows = []\n    for m, res in runs.items():\n        mt = [r[\"metrics\"] for r in res]\n        per_epoch = np.mean([x[\"train_time_s\"] / x[\"epochs_run\"] for x in mt]) / 60\n        rows.append(f\"{MODEL_LABELS[m]} & {mt[0]['parameters'] / 1e6:.2f} & {per_epoch:.2f} & \"\n                    f\"{np.mean([x['inference_ms_per_image'] for x in mt]):.2f} \\\\\\\\\")\n    write(gen / \"table_cost.tex\",\n          \"\\\\begin{tabular}{lccc}\\n\\\\hline\\nModel & Parameters (M) & Training (min/epoch) & Inference (ms/image) \\\\\\\\\\n\\\\hline\\n\"\n          + \"\\n\".join(rows) + \"\\n\\\\hline\\n\\\\end{tabular}\\n\")\n\n\n# --------------------------------------------------------------------------- figures\ndef fig_class_distribution(frame, figdir):\n    fig, ax = plt.subplots(figsize=(TEXT_WIDTH_IN, 2.6))\n    counts = frame.groupby([\"label\", \"split\"]).size().unstack(fill_value=0)[[\"train\", \"val\", \"test\"]]\n    bottom = np.zeros(NUM_CLASSES)\n    for s, c in zip([\"train\", \"val\", \"test\"], PALETTE):\n        v = counts[s].reindex(range(NUM_CLASSES)).values\n        ax.bar(CLASS_CODES, v, bottom=bottom, label=s, color=c)\n        bottom += v\n    ax.set_ylabel(\"Number of images\")\n    ax.set_yscale(\"log\")\n    ax.legend()\n    save_fig(fig, figdir / \"fig_class_distribution\")\n\n\ndef fig_curves(runs, primary, figdir):\n    h = runs[primary][0][\"history\"]\n    ep = [x[\"epoch\"] for x in h]\n    fig, axes = plt.subplots(1, 2, figsize=(TEXT_WIDTH_IN, 2.5))\n    axes[0].plot(ep, [x[\"train_loss\"] for x in h], color=PALETTE[0], label=\"Training\")\n    axes[0].plot(ep, [x[\"val_loss\"] for x in h], color=PALETTE[1], label=\"Validation\")\n    axes[0].set_xlabel(\"Epoch\"); axes[0].set_ylabel(\"Loss\"); axes[0].legend()\n    axes[1].plot(ep, [x[\"val_acc\"] for x in h], color=PALETTE[0], label=\"Accuracy\")\n    axes[1].plot(ep, [x[\"val_balanced_acc\"] for x in h], color=PALETTE[2], label=\"Balanced accuracy\")\n    axes[1].plot(ep, [x[\"val_macro_f1\"] for x in h], color=PALETTE[1], label=\"Macro-F1\")\n    axes[1].set_xlabel(\"Epoch\"); axes[1].set_ylabel(\"Validation score\"); axes[1].legend()\n    fig.tight_layout()\n    save_fig(fig, figdir / \"fig_curves\")\n\n\ndef fig_confusion(runs, figdir, primary=\"inceptionv3\", max_models=6):\n    ranked = sorted(runs, key=lambda m: -np.mean([r[\"metrics\"][\"macro_f1\"] for r in runs[m]]))\n    names = [primary] + [m for m in ranked if m != primary][:max_models - 1]\n    cols = 2\n    rows = int(np.ceil(len(names) / cols))\n    fig, axes = plt.subplots(rows, cols, figsize=(TEXT_WIDTH_IN, 2.2 * rows))\n    axes = np.atleast_1d(axes).ravel()\n    for ax, m in zip(axes, names):\n        cm = np.array(runs[m][0][\"metrics\"][\"confusion_matrix\"], dtype=float)\n        norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)\n        ax.imshow(norm, vmin=0, vmax=1, cmap=\"Blues\")\n        ax.grid(False)\n        for i in range(NUM_CLASSES):\n            for j in range(NUM_CLASSES):\n                ax.text(j, i, f\"{int(cm[i, j])}\", ha=\"center\", va=\"center\", fontsize=5,\n                        color=\"white\" if norm[i, j] > 0.5 else \"black\")\n        ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_CODES, rotation=45, fontsize=5)\n        ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_CODES, fontsize=5)\n        ax.set_title(MODEL_LABELS[m], fontsize=7)\n        ax.set_xlabel(\"Predicted\", fontsize=6); ax.set_ylabel(\"True\", fontsize=6)\n    for ax in axes[len(names):]:\n        ax.axis(\"off\")\n    fig.tight_layout()\n    save_fig(fig, figdir / \"fig_confusion\")\n\n\ndef fig_roc(runs, primary, figdir):\n    d = runs[primary][0][\"preds\"]\n    y, p = d[\"y_true\"], d[\"probs\"]\n    fig, ax = plt.subplots(figsize=(TEXT_WIDTH_IN * 0.7, 3.2))\n    for k, c in enumerate(PALETTE):\n        if (y == k).sum() == 0:\n            continue\n        fpr, tpr, _ = roc_curve((y == k).astype(int), p[:, k])\n        auc = runs[primary][0][\"metrics\"][\"per_class\"][\"auroc\"][k]\n        ax.plot(fpr, tpr, color=c, label=f\"{CLASS_CODES[k]} (AUC {auc:.3f})\")\n    ax.plot([0, 1], [0, 1], \"k--\", lw=0.8)\n    ax.set_xlabel(\"False positive rate\"); ax.set_ylabel(\"True positive rate\"); ax.legend(loc=\"lower right\")\n    save_fig(fig, figdir / \"fig_roc\")\n\n\ndef fig_comparison(runs, figdir):\n    keys = [(\"accuracy\", \"Accuracy\"), (\"balanced_accuracy\", \"Balanced accuracy\"), (\"macro_f1\", \"Macro-F1\")]\n    names = list(runs)\n    x = np.arange(len(names))\n    w = 0.26\n    fig, ax = plt.subplots(figsize=(TEXT_WIDTH_IN, 2.8))\n    for i, (k, lab) in enumerate(keys):\n        vals = [np.mean([r[\"metrics\"][k] for r in runs[m]]) for m in names]\n        err = [np.std([r[\"metrics\"][k] for r in runs[m]], ddof=1) if len(runs[m]) > 1 else 0 for m in names]\n        ax.bar(x + (i - 1) * w, vals, w, yerr=err if any(err) else None, capsize=2, label=lab, color=PALETTE[i])\n    ax.set_xticks(x); ax.set_xticklabels([MODEL_LABELS[m] for m in names], rotation=20)\n    ax.set_ylabel(\"Score\"); ax.set_ylim(0, 1); ax.legend(ncol=3, loc=\"upper left\")\n    fig.tight_layout()\n    save_fig(fig, figdir / \"fig_comparison\")\n\n\ndef fig_samples(frame, figdir, n=5):\n    from PIL import Image\n    fig, axes = plt.subplots(NUM_CLASSES, n, figsize=(TEXT_WIDTH_IN, TEXT_WIDTH_IN * NUM_CLASSES / n * 0.9))\n    for k, code in enumerate(CLASS_CODES):\n        sub = frame[frame.label == k].sample(n, random_state=1)\n        for j, path in enumerate(sub[\"path\"]):\n            axes[k, j].imshow(Image.open(path).convert(\"RGB\"))\n            axes[k, j].axis(\"off\")\n        axes[k, 0].set_title(code, fontsize=8, loc=\"left\")\n    fig.tight_layout()\n    save_fig(fig, figdir / \"fig_samples\")\n\n\n# --------------------------------------------------------------------------- numbers and text\ndef choose_primary(runs):\n    \"\"\"Model with the highest mean (over seeds) of the best validation macro-F1; no test data involved.\"\"\"\n    def val_f1(m):\n        return float(np.mean([max(h[\"val_macro_f1\"] for h in r[\"history\"]) for r in runs[m]]))\n    return max(runs, key=val_f1)\n\n\nPREFIX = {\"simple_cnn\": \"Cnn\", \"cnn_lstm\": \"Lstm\", \"mobilenetv2\": \"Mob\", \"inceptionv3\": \"Inc\",\n          \"efficientnet_b0\": \"Eff\", \"efficientnet_v2_s\": \"Effv\", \"convnext_tiny\": \"Cnx\", \"swin_t\": \"Swin\",\n          \"vit_small\": \"Vit\", \"dinov2_small\": \"Dino\"}\n\n\ndef write_numbers_and_text(runs, primary, frame, gen):\n    \"\"\"Write LaTeX macros and short results paragraphs that are computed directly from the saved runs.\"\"\"\n    def mean(m, k):\n        return float(np.mean([r[\"metrics\"][k] for r in runs[m]]))\n\n    pm = runs[primary][0][\"metrics\"]\n    n_seeds = len(runs[primary])\n    base = pm[\"majority_baseline\"]\n    macros = {\n        \"PrimaryName\": MODEL_LABELS[primary], \"PrimaryAcc\": f\"{mean(primary, 'accuracy'):.3f}\",\n        \"PrimaryBalAcc\": f\"{mean(primary, 'balanced_accuracy'):.3f}\",\n        \"PrimaryMacroF\": f\"{mean(primary, 'macro_f1'):.3f}\", \"PrimaryMCC\": f\"{mean(primary, 'mcc'):.3f}\",\n        \"PrimaryAUC\": f\"{mean(primary, 'auroc_macro'):.3f}\", \"BaselineAcc\": f\"{base['accuracy']:.3f}\",\n        \"NTest\": str(int((frame.split == 'test').sum())), \"NTrain\": str(int((frame.split == 'train').sum())),\n        \"NVal\": str(int((frame.split == 'val').sum())), \"NSeeds\": str(n_seeds),\n    }\n    mel_idx = CLASS_CODES.index(\"mel\")\n    for m in runs:\n        pf = PREFIX[m]\n        macros[pf + \"Acc\"] = f\"{mean(m, 'accuracy'):.3f}\"\n        macros[pf + \"BalAcc\"] = f\"{mean(m, 'balanced_accuracy'):.3f}\"\n        macros[pf + \"MacroF\"] = f\"{mean(m, 'macro_f1'):.3f}\"\n        macros[pf + \"MCC\"] = f\"{mean(m, 'mcc'):.3f}\"\n        macros[pf + \"AUC\"] = f\"{mean(m, 'auroc_macro'):.3f}\"\n        macros[pf + \"MelSens\"] = f\"{np.mean([r['metrics']['per_class']['recall'][mel_idx] for r in runs[m]]):.3f}\"\n        macros[pf + \"MelSpec\"] = f\"{np.mean([r['metrics']['per_class']['specificity'][mel_idx] for r in runs[m]]):.3f}\"\n    macros[\"OverallNote\"] = (\"single training run per model\" if n_seeds == 1\n                             else \"mean $\\\\pm$ standard deviation over seeds\")\n    lines = [f\"\\\\newcommand{{\\\\{k}}}{{{v}}}\" for k, v in macros.items()]\n    write(gen / \"numbers.tex\", \"\\n\".join(lines) + \"\\n\")\n\n    order = sorted(runs, key=lambda m: -mean(m, \"macro_f1\"))\n    rank = order.index(primary) + 1\n    rank_txt = \", \".join(f\"{MODEL_LABELS[m]} ({mean(m, 'macro_f1'):.3f})\" for m in order)\n    verdict = (\"exceeds\" if mean(primary, \"accuracy\") > base[\"accuracy\"] else \"does not exceed\")\n    overall = (\n        f\"On the held-out test partition of {macros['NTest']} images, {MODEL_LABELS[primary]} reached an accuracy of \"\n        f\"{macros['PrimaryAcc']}, a balanced accuracy of {macros['PrimaryBalAcc']}, a macro-averaged F1-score of \"\n        f\"{macros['PrimaryMacroF']} and a Matthews correlation coefficient of {macros['PrimaryMCC']} \"\n        f\"({'single training run' if n_seeds == 1 else 'mean over ' + str(n_seeds) + ' training seeds'}). A classifier that always predicts the most frequent training class \"\n        f\"reaches an accuracy of {macros['BaselineAcc']} on the same images, so the accuracy of {MODEL_LABELS[primary]} \"\n        f\"{verdict} this reference value. Ranked by macro-F1, the models order as follows: {rank_txt}. \"\n        f\"{MODEL_LABELS[primary]} ranks {rank} of {len(order)}.\")\n    write(gen / \"results_overall.tex\", overall + \"\\n\")\n\n    res = runs[primary]\n    rec = np.mean([r[\"metrics\"][\"per_class\"][\"recall\"] for r in res], axis=0)\n    spec = np.mean([r[\"metrics\"][\"per_class\"][\"specificity\"] for r in res], axis=0)\n    hi, lo = int(rec.argmax()), int(rec.argmin())\n    mel = CLASS_CODES.index(\"mel\")\n    perclass = (\n        f\"For {MODEL_LABELS[primary]}, sensitivity was highest for {CLASS_CODES[hi]} ({rec[hi]:.3f}) and lowest for \"\n        f\"{CLASS_CODES[lo]} ({rec[lo]:.3f}). For melanoma, the sensitivity was {rec[mel]:.3f} and the \"\n        f\"specificity was {spec[mel]:.3f}.\")\n    write(gen / \"results_perclass.tex\", perclass + \"\\n\")\n\n    ci = pm[\"ci95\"]\n    ci_txt = (\n        f\"For {MODEL_LABELS[primary]}, the 95\\\\% bootstrap interval was \"\n        f\"[{ci['accuracy'][0]:.3f}, {ci['accuracy'][1]:.3f}] for accuracy, \"\n        f\"[{ci['balanced_accuracy'][0]:.3f}, {ci['balanced_accuracy'][1]:.3f}] for balanced accuracy and \"\n        f\"[{ci['macro_f1'][0]:.3f}, {ci['macro_f1'][1]:.3f}] for macro-F1.\")\n    write(gen / \"results_ci.tex\", ci_txt + \"\\n\")\n\n    fastest = min(runs, key=lambda m: np.mean([r[\"metrics\"][\"inference_ms_per_image\"] for r in runs[m]]))\n    cost = (\n        f\"{MODEL_LABELS[primary]} has {pm['parameters'] / 1e6:.2f} million parameters and needed \"\n        f\"{np.mean([r['metrics']['inference_ms_per_image'] for r in res]):.2f} ms per image at inference on the \"\n        f\"evaluation hardware ({pm.get('device', 'unspecified device')}). The fastest model was {MODEL_LABELS[fastest]} with \"\n        f\"{np.mean([r['metrics']['inference_ms_per_image'] for r in runs[fastest]]):.2f} ms per image.\")\n    write(gen / \"results_cost.tex\", cost + \"\\n\")\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--out-dir\", default=\"outputs\")\n    ap.add_argument(\"--paper-dir\", default=\"paper\")\n    ap.add_argument(\"--primary\", default=\"auto\",\n                    help=\"Model for the detailed analysis; auto = best mean validation macro-F1\")\n    ap.add_argument(\"--data-dir\", default=None, help=\"Optional: HAM10000 folder, used for the sample-image figure\")\n    args = ap.parse_args()\n\n    gen, figdir = Path(args.paper_dir) / \"generated\", Path(args.paper_dir) / \"figures\"\n    frame = pd.read_csv(Path(args.out_dir) / \"splits.csv\")\n    runs = load_runs(args.out_dir)\n    if args.primary == \"auto\":\n        args.primary = choose_primary(runs)\n        print(\"Primary model (best validation macro-F1):\", args.primary)\n    if args.primary not in runs:\n        raise SystemExit(f\"No results found for primary model '{args.primary}' in {args.out_dir}\")\n\n    table_dataset(frame, gen); table_overall(runs, gen); table_perclass(runs, args.primary, gen)\n    table_ci(runs, gen); table_cost(runs, gen)\n    fig_class_distribution(frame, figdir); fig_curves(runs, args.primary, figdir)\n    fig_confusion(runs, figdir, args.primary); fig_roc(runs, args.primary, figdir); fig_comparison(runs, figdir)\n    if args.data_dir and \"path\" in frame:\n        fig_samples(frame, figdir)\n    write_numbers_and_text(runs, args.primary, frame, gen)\n    write(gen / \"primary.txt\", args.primary + \"\\n\")\n    summary = {m: {k: float(np.mean([r[\"metrics\"][k] for r in res]))\n                   for k in (\"accuracy\", \"balanced_accuracy\", \"macro_f1\", \"mcc\", \"auroc_macro\")}\n               for m, res in runs.items()}\n    write(gen / \"summary.json\", json.dumps(summary, indent=2))\n    print(\"Report written to\", args.paper_dir)\n\n\nif __name__ == \"__main__\":\n    main()\n",
"src/train.py": "\"\"\"Train and evaluate one model on HAM10000 with a lesion-level split.\n\nExample:\n    python -m src.train --data-dir data/HAM10000 --model inceptionv3 --seeds 0 1 2\n\"\"\"\nimport argparse\nimport json\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\nfrom tqdm import tqdm\n\nfrom .common import MODEL_ORDER, NUM_CLASSES\nfrom .data import LesionDataset, build_transforms, load_metadata, make_splits\nfrom .evaluate import bootstrap_ci, compute_metrics, majority_baseline\nfrom .models import INPUT_SIZE, PRETRAINED, build_model, count_parameters\n\n\nclass FocalLoss(nn.Module):\n    def __init__(self, weight=None, gamma=2.0):\n        super().__init__()\n        self.weight, self.gamma = weight, gamma\n\n    def forward(self, logits, target):\n        logp = F.log_softmax(logits, dim=1)\n        ce = F.nll_loss(logp, target, weight=self.weight, reduction=\"none\")\n        pt = logp.gather(1, target[:, None]).squeeze(1).exp()\n        return ((1 - pt) ** self.gamma * ce).mean()\n\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef class_weights(labels, power=0.5):\n    counts = np.bincount(labels, minlength=NUM_CLASSES).astype(float)\n    w = (counts.sum() / (NUM_CLASSES * counts)) ** power\n    return torch.tensor(w / w.mean(), dtype=torch.float32)\n\n\n@torch.no_grad()\ndef predict(model, loader, device):\n    model.eval()\n    probs, ys = [], []\n    for x, y in loader:\n        x = x.to(device, non_blocking=True)\n        with torch.autocast(device_type=device.type, enabled=device.type == \"cuda\"):\n            logits = model(x)\n        probs.append(F.softmax(logits.float(), dim=1).cpu().numpy())\n        ys.append(y.numpy())\n    return np.concatenate(ys), np.concatenate(probs)\n\n\ndef make_optimizer(model, lr, wd):\n    params = [p for p in model.parameters() if p.requires_grad]\n    return torch.optim.AdamW(params, lr=lr, weight_decay=wd)\n\n\ndef run_one(args, seed, frame, out_root):\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    set_seed(seed)\n    size = INPUT_SIZE[args.model]\n    pretrained = args.model in PRETRAINED\n    tr, va, te = (frame[frame.split == s] for s in (\"train\", \"val\", \"test\"))\n    kw = dict(batch_size=args.batch_size, num_workers=args.workers, pin_memory=True)\n    train_loader = DataLoader(LesionDataset(tr, build_transforms(size, True)), shuffle=True, drop_last=True, **kw)\n    val_loader = DataLoader(LesionDataset(va, build_transforms(size, False)), **kw)\n    test_loader = DataLoader(LesionDataset(te, build_transforms(size, False)), **kw)\n\n    model, head = build_model(args.model, pretrained=pretrained)\n    model.to(device)\n    w = class_weights(tr[\"label\"].values).to(device) if args.loss != \"ce\" else None\n    criterion = FocalLoss(w) if args.loss == \"focal\" else nn.CrossEntropyLoss(weight=w)\n    scaler = torch.amp.GradScaler(enabled=device.type == \"cuda\")\n\n    out_dir = Path(out_root) / args.model / f\"seed{seed}\"\n    out_dir.mkdir(parents=True, exist_ok=True)\n    history, best_f1, best_state, bad = [], -1.0, None, 0\n    warm = args.warmup_epochs if pretrained else 0\n    opt = sched = None\n    t0 = time.time()\n\n    for epoch in range(args.epochs):\n        if epoch == 0 and warm > 0:                      # stage 1: classifier head only\n            for p in model.parameters():\n                p.requires_grad = False\n            for p in head.parameters():\n                p.requires_grad = True\n            opt, sched = make_optimizer(model, args.lr_head, args.weight_decay), None\n        if epoch == warm:                                # stage 2: full network\n            for p in model.parameters():\n                p.requires_grad = True\n            lr = args.lr_finetune if pretrained else args.lr_head\n            opt = make_optimizer(model, lr, args.weight_decay)\n            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, args.epochs - warm))\n\n        model.train()\n        run_loss, seen, correct = 0.0, 0, 0\n        for x, y in tqdm(train_loader, desc=f\"{args.model} s{seed} e{epoch + 1}\", leave=False):\n            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, enabled=device.type == \"cuda\"):\n                logits = model(x)\n                loss = criterion(logits.float(), y)\n            scaler.scale(loss).backward()\n            scaler.step(opt)\n            scaler.update()\n            run_loss += loss.item() * len(y)\n            seen += len(y)\n            correct += (logits.argmax(1) == y).sum().item()\n        if sched is not None:\n            sched.step()\n\n        yv, pv = predict(model, val_loader, device)\n        vm = compute_metrics(yv, pv)\n        val_loss = float(F.nll_loss(torch.log(torch.tensor(pv).clamp_min(1e-8)), torch.tensor(yv)))\n        history.append({\"epoch\": epoch + 1, \"train_loss\": run_loss / seen, \"train_acc\": correct / seen,\n                        \"val_loss\": val_loss, \"val_acc\": vm[\"accuracy\"], \"val_macro_f1\": vm[\"macro_f1\"],\n                        \"val_balanced_acc\": vm[\"balanced_accuracy\"]})\n        print(json.dumps(history[-1]))\n        if vm[\"macro_f1\"] > best_f1:                     # model selection on validation macro-F1\n            best_f1, bad = vm[\"macro_f1\"], 0\n            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        else:\n            bad += 1\n            if epoch >= warm and bad >= args.patience:\n                print(\"Early stopping.\")\n                break\n    train_time = time.time() - t0\n\n    model.load_state_dict(best_state)\n    torch.save(best_state, out_dir / \"best.pt\")\n    t1 = time.time()\n    yt, pt = predict(model, test_loader, device)\n    infer_ms = 1000 * (time.time() - t1) / len(yt)\n\n    metrics = compute_metrics(yt, pt)\n    metrics[\"ci95\"] = bootstrap_ci(yt, pt)\n    metrics[\"majority_baseline\"] = majority_baseline(tr[\"label\"].values, yt)\n    metrics.update({\"model\": args.model, \"seed\": seed, \"parameters\": count_parameters(model),\n                    \"train_time_s\": train_time, \"epochs_run\": len(history),\n                    \"inference_ms_per_image\": infer_ms, \"loss\": args.loss,\n                    \"device\": torch.cuda.get_device_name(0) if device.type == \"cuda\" else \"cpu\",\n                    \"n_train\": len(tr), \"n_val\": len(va), \"n_test\": len(te)})\n    (out_dir / \"metrics.json\").write_text(json.dumps(metrics, indent=2))\n    (out_dir / \"history.json\").write_text(json.dumps(history, indent=2))\n    np.savez_compressed(out_dir / \"test_predictions.npz\", y_true=yt, probs=pt)\n    print(f\"[{args.model} seed {seed}] acc={metrics['accuracy']:.4f} \"\n          f\"bal_acc={metrics['balanced_accuracy']:.4f} macro_f1={metrics['macro_f1']:.4f}\")\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\"--data-dir\", required=True, help=\"Folder with HAM10000_metadata.csv and the .jpg images\")\n    ap.add_argument(\"--out-dir\", default=\"outputs\")\n    ap.add_argument(\"--model\", required=True,\n                    choices=MODEL_ORDER)\n    ap.add_argument(\"--seeds\", type=int, nargs=\"+\", default=[0, 1, 2])\n    ap.add_argument(\"--split-seed\", type=int, default=42)\n    ap.add_argument(\"--epochs\", type=int, default=30)\n    ap.add_argument(\"--warmup-epochs\", type=int, default=3)\n    ap.add_argument(\"--patience\", type=int, default=7)\n    ap.add_argument(\"--batch-size\", type=int, default=32)\n    ap.add_argument(\"--lr-head\", type=float, default=1e-3)\n    ap.add_argument(\"--lr-finetune\", type=float, default=1e-4)\n    ap.add_argument(\"--weight-decay\", type=float, default=1e-4)\n    ap.add_argument(\"--loss\", choices=[\"ce\", \"weighted_ce\", \"focal\"], default=\"weighted_ce\")\n    ap.add_argument(\"--workers\", type=int, default=4)\n    args = ap.parse_args()\n\n    out_root = Path(args.out_dir)\n    out_root.mkdir(parents=True, exist_ok=True)\n    split_file = out_root / \"splits.csv\"\n    if split_file.exists():\n        import pandas as pd\n        frame = pd.read_csv(split_file)\n    else:\n        frame = make_splits(load_metadata(args.data_dir), seed=args.split_seed)\n        frame.to_csv(split_file, index=False)\n    for seed in args.seeds:\n        run_one(args, seed, frame, out_root)\n\n\nif __name__ == \"__main__\":\n    main()\n"
}
os.makedirs('/kaggle/working/skin/src', exist_ok=True)
for path, text in FILES.items():
    open('/kaggle/working/skin/' + path, 'w').write(text)
os.chdir('/kaggle/working/skin')
print('Wrote', len(FILES), 'files to', os.getcwd())

In [ ]:
import glob, subprocess, sys, json
import numpy as np, pandas as pd
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True)
meta_path = glob.glob('/kaggle/input/**/HAM10000_metadata.csv', recursive=True)
assert meta_path, 'Add the dataset "Skin Cancer MNIST: HAM10000" as an input to this notebook'
DATA_DIR = os.path.dirname(meta_path[0])
import torch; print('Data:', DATA_DIR, '| GPU available:', torch.cuda.is_available())

# 1 Data Exploration

In [ ]:
metadata = pd.read_csv(meta_path[0])
print(metadata.head())
print('\nClass proportions:\n', (metadata['dx'].value_counts() / len(metadata)).round(4))
per_lesion = metadata['lesion_id'].value_counts()
print('\nImages:', len(metadata), '| lesions:', len(per_lesion), '| lesions with more than one image:', int((per_lesion > 1).sum()))

# 2 Split the Data (lesion-grouped, stratified, about 70/10/20)

In [ ]:
from src.data import load_metadata, make_splits
from src.common import CLASS_CODES
frame = make_splits(load_metadata(DATA_DIR), seed=42)
os.makedirs('outputs', exist_ok=True)
frame.to_csv('outputs/splits.csv', index=False)          # every model and seed uses this partition
print(pd.crosstab(frame['dx'], frame['split'])[['train', 'val', 'test']])
leak = frame.groupby('lesion_id')['split'].nunique().max()
print('\nMax number of partitions per lesion (must be 1):', leak)

# 3 Train the Models

In [ ]:
# Choose a preset. 'quick' = 4 models, 1 seed. 'full' = all 10 models, 3 seeds (use Commit / several sessions).
PRESET = 'quick'
PRESETS = {
    'quick': (['inceptionv3', 'mobilenetv2', 'convnext_tiny', 'simple_cnn'], [0]),
    'full':  (['simple_cnn', 'cnn_lstm', 'mobilenetv2', 'inceptionv3', 'efficientnet_b0',
               'efficientnet_v2_s', 'convnext_tiny', 'swin_t', 'vit_small', 'dinov2_small'], [0, 1, 2]),
}
MODELS, SEEDS = PRESETS[PRESET]
EPOCHS = 30
for seed in SEEDS:
    for m in MODELS:
        if os.path.exists(f'outputs/{m}/seed{seed}/metrics.json'):
            print('skip (already done):', m, seed); continue
        subprocess.run([sys.executable, '-m', 'src.train', '--data-dir', DATA_DIR, '--model', m,
                        '--seeds', str(seed), '--epochs', str(EPOCHS), '--workers', '2'], check=True)

# 4 Evaluate the Models

In [ ]:
rows = []
for f in sorted(glob.glob('outputs/*/seed*/metrics.json')):
    r = json.load(open(f))
    rows.append({'model': r['model'], 'seed': r['seed'], 'accuracy': r['accuracy'], 'balanced_acc': r['balanced_accuracy'],
                 'macro_f1': r['macro_f1'], 'mcc': r['mcc'], 'auroc': r['auroc_macro'],
                 'majority_baseline_acc': r['majority_baseline']['accuracy']})
results = pd.DataFrame(rows).round(3)
print(results.to_string(index=False))

In [ ]:
# Confusion matrix of one model (first seed) from the saved test predictions
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
SHOW = 'inceptionv3'
d = np.load(f'outputs/{SHOW}/seed0/test_predictions.npz')
cm = confusion_matrix(d['y_true'], d['probs'].argmax(1), labels=range(7))
plt.imshow(cm / cm.sum(1, keepdims=True), cmap=plt.cm.Blues); plt.colorbar()
plt.xticks(range(7), CLASS_CODES, rotation=45); plt.yticks(range(7), CLASS_CODES)
plt.xlabel('predicted'); plt.ylabel('true'); plt.title(f'{SHOW}: row-normalised confusion matrix'); plt.tight_layout(); plt.show()

# 5 Figures and tables for the paper

In [ ]:
# Builds paper/figures and paper/generated from the saved results. The detailed-analysis model is chosen automatically
# (best validation macro-F1). Grad-CAM is run for that model when its checkpoint exists.
os.makedirs('paper', exist_ok=True)
subprocess.run([sys.executable, '-m', 'src.diagrams'], check=False)
subprocess.run([sys.executable, '-m', 'src.report', '--out-dir', 'outputs', '--paper-dir', 'paper',
                '--data-dir', DATA_DIR], check=False)
primary = open('paper/generated/primary.txt').read().strip()
from src.gradcam import TARGET_LAYER
if primary in TARGET_LAYER and os.path.exists(f'outputs/{primary}/seed0/best.pt'):
    subprocess.run([sys.executable, '-m', 'src.gradcam', '--data-dir', DATA_DIR, '--splits', 'outputs/splits.csv',
                    '--model', primary, '--checkpoint', f'outputs/{primary}/seed0/best.pt',
                    '--out', 'paper/figures/fig_gradcam'], check=False)


In [ ]:
# Package results (without the large checkpoints) -> download outputs_results.zip from the Output tab
import shutil
shutil.rmtree('outputs_light', ignore_errors=True)
shutil.copytree('outputs', 'outputs_light', ignore=shutil.ignore_patterns('best.pt'))
for d in ('figures', 'generated'):
    if os.path.isdir(f'paper/{d}'):
        shutil.copytree(f'paper/{d}', f'outputs_light/paper_{d}', dirs_exist_ok=True)
shutil.make_archive('/kaggle/working/outputs_results', 'zip', 'outputs_light')
print('Created /kaggle/working/outputs_results.zip')